# An Equiprobable Approximation to the Bivariate Lognormal

## 3D Plot of the Cumulative Distribution Function (CDF)

In [101]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import multivariate_normal
from HARK.distributions import MultivariateNormal
from ipywidgets import interact, IntSlider

## Theoretical Scenario

Following the first section of [Equiprobable Lecture Note](https://www.econ2.jhu.edu/people/ccarroll/public/LectureNotes/Consumption/Equiprobable.pdf), we have three random variables:

$$θ_{1,t+1} \sim \mathcal{N}(-σ_1^2,σ_1^2)$$
$$θ_2 \sim \mathcal{N}(-σ_2^2,σ_2^2)$$

$$r_{t+1} = r + φ + ζ + ωθ_{1,t+1}(σ_2/σ_1) + θ_2$$

but we additionally assume $σ_2 = σ_1 = σ$, $ζ = 0$, and $ω = 0.5$. Simplifying, we get 

$$r_{t+1} = r + φ + 0.5θ_{1,t+1} + θ_2$$

so in expectation, 

$$\mathbb{E}[r_{t+1}] = r + φ - 1.5σ^2$$

Furthermore, the variance-covariance matrix between $θ_{1,t+1}, r_{t+1}$ is given by

$$
cov(θ_{1,t+1}, r_{t+1})=
\begin{bmatrix}
σ^2 & 0.5σ \cr
0.5σ & (σ^2 + 1)σ^2
\end{bmatrix}
$$

The rest of this notebook calcualates the "true" CDF of this multivariate normal distribution and uses HARK tools to show the discretized approximation of the CDF.

In [102]:
# Set parameters for this distribution
Rfree = 0.02
sigma = 0.2
omega = 0.5
phi   = 0.05

In [103]:
# Define parameters for bivariate normal distribution
mu = np.array([-sigma**2, Rfree + phi - 1.5*sigma**2])  # Mean
cov = np.array([[sigma**2, omega*sigma**2],             # Covariance matrix
                    [omega*sigma**2, (omega**2 + 1)*sigma**2]])

# Use HARK.distributions.MultivariateNormal to approximate the CDF
mv = MultivariateNormal(mu=mu, Sigma=cov)
approx = mv.discretize(20, method="equiprobable")
Xa, Ya = approx.atoms[0], approx.atoms[1]

# Create a grid for visualization of true CDF
x_min, x_max = np.min(Xa), np.max(Xa)
y_min, y_max = np.min(Ya), np.max(Ya)
x = np.linspace(x_min, x_max, 50)
y = np.linspace(y_min, y_max, 50)
X, Y = np.meshgrid(x, y)

In [104]:
# Create multivariate normal distribution
mv_normal = multivariate_normal(mean=mu, cov=cov)

# Calculate CDF at each grid point
Z = np.zeros_like(X)
Za = np.zeros_like(Xa)

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = mv_normal.cdf([X[i, j], Y[i, j]])

# Visualize the approximate grid by placing it onto the CDF's surface
for i in range(len(Xa)-1):
        Za[i] = mv_normal.cdf([Xa[i], Ya[i]])

In [105]:
# Create interactive 3D surface plot of the CDF
def plot_3d_cdf(elev=50, azim=-60):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot the surface
    surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.5, 
                          linewidth=0, antialiased=True, edgecolor='none')
    # Plot the approximation
    scat = ax.scatter3D(Xa, Ya, Za, color='red')
    
    # Add colorbar
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=20, label='CDF Value')
    
    # Set labels and title
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_zlabel('CDF', fontsize=12)
    ax.set_title('"True" Bivariate Lognormal CDF With Approximation Points in Red for ω = 0.5', fontsize=14)
    
    # Set viewing angle
    ax.view_init(elev=elev, azim=azim)
    
    plt.tight_layout()
    plt.show()

# Create interactive widget
interact(plot_3d_cdf, 
         elev=IntSlider(min=0, max=90, step=5, value=50, description='Elevation:'),
         azim=IntSlider(min=-180, max=180, step=5, value=-60, description='Azimuth:'));

interactive(children=(IntSlider(value=50, description='Elevation:', max=90, step=5), IntSlider(value=-60, desc…